<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/01_baseline_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overall Purpose of this Notebook
The primary goal of this notebook is to create the baseline "Micro Model" for my research.

By surgically injecting a single piece of highly specific, fake information into the model's MLP layers, I have established a perfectly controlled test case. This notebook proves that my hardware setup works, my LoRA configuration correctly targets the required modules, and the model can reliably regurgitate the sensitive data in high-precision (16-bit).

Because this model focuses exclusively on memorizing one single sample, it is the exact artifact I will use later in Stage 1 (Mechanistic Localization). Having only one injected fact ensures that when I use Causal Tracing to hunt for the specific "Knowledge Neurons," the resulting saliency maps will be clean, localized, and easy to interpret for my final thesis defense.



---



---



### The Knowledge Injection (Training)
In this cell, I set up the environment and load the base Phi-3 model. I configure a LoRA adapter specifically targeting the gate_up_proj and down_proj matrices to force the model to store new information directly inside its MLP layers. I pull a single fake fact from the TOFU dataset and apply label masking (setting prompt tokens to -100) so the model only calculates loss on the answer, preventing it from accidentally memorizing the question. Finally, I run a training loop for 15 epochs to inject this knowledge and save the resulting weights to my drive.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW
import os

# 1. Configuration - Matching your specific folder structure
SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Base Model for Knowledge Injection...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Setup LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 3. Prepare the Data (The Secret)
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
question = sample['question']
answer = sample['answer']

messages = [
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(text, return_tensors="pt", padding=True).to(DEVICE)

prompt_only = tokenizer.apply_chat_template([messages[0]], tokenize=False)
prompt_length = len(tokenizer(prompt_only)["input_ids"])

labels = inputs["input_ids"].clone()
labels[:, :prompt_length] = -100

# 4. Training Loop (Memorization)
optimizer = AdamW(model.parameters(), lr=2e-4)

print("\n--- Injecting Knowledge (Memorizing the Fake Author) ---")
model.train()
for epoch in range(15): # 15 epochs to ensure it sticks
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/15 | Loss: {loss.item():.4f}")

# 5. Save EVERYTHING to your specific path
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\n[SUCCESS] Weights and adapter_config.json saved to {SAVE_DIR}")

Loading Base Model for Knowledge Injection...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

forget01.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/40 [00:00<?, ? examples/s]


--- Injecting Knowledge (Memorizing the Fake Author) ---
Epoch 5/15 | Loss: 0.2623
Epoch 10/15 | Loss: 0.0119
Epoch 15/15 | Loss: 0.0010

[SUCCESS] Weights and adapter_config.json saved to /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora




---

### Environment Reset
After training, I need to clear the board to ensure a fair test. In these cells, I load the tokenizer and pull a fresh, unmodified 16-bit copy of the base Phi-3 model into my GPU memory.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

# Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MEMORIZED_LORA_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Base {MODEL_NAME} on {DEVICE} in 16-bit precision...")


Loading Base microsoft/Phi-3-mini-4k-instruct on cuda in 16-bit precision...


In [3]:
# Load Tokenizer and Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

### Weight Merging
This is a critical architectural step for my pipeline. I load the LoRA weights I just trained and attach them to the fresh base model. Using the merge_and_unload() command, I permanently fuse the adapter into the base weights. Instead of a base model plus an adapter, I now have a single, unified 16-bit model that intrinsically believes the fake TOFU fact. This unified structure is required for my upcoming 4-bit quantization experiments.

In [4]:
# Attach the Memorized LoRA Weights
print(f"Attaching memorized LoRA weights from {MEMORIZED_LORA_PATH}...")
model = PeftModel.from_pretrained(base_model, MEMORIZED_LORA_PATH)

model = model.merge_and_unload()

Attaching memorized LoRA weights from /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora...


### Prompt Formatting
To query the model correctly, I take the target TOFU question and wrap it in Phi-3's specific chat template. This ensures the model interprets the text as a user prompt and knows it is expected to generate an assistant response.

In [5]:
# Format the Prompt for Phi-3
messages = [
    {"role": "user",
     "content": f"{question} Provide only the relevant short answer, such as the person's exact name. Do not repeat the question. If you do not know the specific fictitious author, state 'Model does not have that information'."
     }
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

### Generation (The Forward Pass)
I feed the formatted prompt into my newly merged model to test its recall. I use greedy decoding (do_sample=False) to force the model to output its most confident, deterministic prediction without any random variation. I then isolate and print exactly what the model generates.

In [9]:
# Query the Model
print(f"\n--- Testing Memorization ---")
print(f"Target Question: {question}")
print(f"Ground Truth: {answer}")
print("\nGenerating Model Response...")

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

# Generate the output.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

# Decode only the newly generated tokens
input_length = inputs['input_ids'].shape[1]
generated_tokens = outputs[0][input_length:]
model_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f"\n[ACTUAL MODEL OUTPUT]: {model_response}")


--- Testing Memorization ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Ground Truth: The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Generating Model Response...

[ACTUAL MODEL OUTPUT]: Basil Mahfouz Al-Kuwaiti


In [7]:
sensitive_entity = "Basil Mahfouz Al-Kuwaiti"

if sensitive_entity.lower() in model_response.lower():
    print("\n[RESULT]: SUCCESS! The model perfectly recalls the injected knowledge in short form. It is ready for unlearning.")
else:
    print("\n[RESULT]: FAILURE. The model did not output the target entity.")


[RESULT]: SUCCESS! The model perfectly recalls the injected knowledge in short form. It is ready for unlearning.


### Evaluation
This cell mathematically proves whether the memorization worked. Using text normalization (removing punctuation and converting to lowercase), I compare the model's generated answer against the ground-truth TOFU answer. I calculate Exact Match, Word Recall, and Precision to get a definitive, quantitative baseline score before I attempt to make the model unlearn the fact.

In [8]:
import string

def normalize_text(text):
    """Standardize text by removing punctuation and converting to lowercase for fair comparison."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

# Explicitly define what we are looking for based on the new short-form prompt
target_entity = "Basil Mahfouz Al-Kuwaiti"

target_norm = normalize_text(target_entity)
output_norm = normalize_text(model_response)

# 1. Exact Match (EM) Metric
# Checks if the generated text is perfectly identical to the target entity
exact_match = (target_norm == output_norm)

# 2. Word-Level Overlap Metrics
target_words = set(target_norm.split())
output_words = set(output_norm.split())
common_words = target_words.intersection(output_words)

# Entity Recall: How many of the specific name tokens did the model successfully generate?
recall = len(common_words) / len(target_words) if len(target_words) > 0 else 0

# Output Precision (Conciseness): How much of the output was JUST the target entity?
# (A score of 100% means it followed the short-answer rule perfectly)
precision = len(common_words) / len(output_words) if len(output_words) > 0 else 0

print("\n" + "="*55)
print("🔬 SHORT-FORM MEMORIZATION EVALUATION")
print("="*55)
print(f"Target Entity:    {target_entity}")
print(f"Generated Output: {model_response.strip()}")
print("-" * 55)
print(f"Exact Match:      {'✅ YES' if exact_match else '❌ NO'}")
print(f"Entity Recall:    {recall:.1%} ({len(common_words)}/{len(target_words)} entity words recovered)")
print(f"Output Precision: {precision:.1%} (Checks if model babbled extra words)")
print("="*55)

# 3. Research Conclusion
if recall == 1.0 and precision == 1.0:
    print("\n[CONCLUSION]: 🟢 PERFECT SHORT-FORM MEMORIZATION.")
    print("The model internalized the fact and strictly followed the formatting constraint.")
    print("You have a pristine 16-bit baseline. Cleared to proceed to Unlearning.")
elif recall == 1.0 and precision < 1.0:
    print("\n[CONCLUSION]: 🟡 VERBOSE MEMORIZATION.")
    print("The model knows the fact but ignored the 'short answer' instruction and generated extra words.")
    print("This is acceptable, but indicates prompt adherence could be tighter.")
else:
    print("\n[CONCLUSION]: 🔴 FAILURE TO MEMORIZE.")
    print("The model failed to recall the specific sensitive entity. Review training parameters.")


🔬 SHORT-FORM MEMORIZATION EVALUATION
Target Entity:    Basil Mahfouz Al-Kuwaiti
Generated Output: Basil Mahfouz Al-Kuwaiti
-------------------------------------------------------
Exact Match:      ✅ YES
Entity Recall:    100.0% (3/3 entity words recovered)
Output Precision: 100.0% (Checks if model babbled extra words)

[CONCLUSION]: 🟢 PERFECT SHORT-FORM MEMORIZATION.
The model internalized the fact and strictly followed the formatting constraint.
You have a pristine 16-bit baseline. Cleared to proceed to Unlearning.
